# S9 - Observabilidad con Grafana y costos

**Proyecto Sello — Equipo LLSW3 — Telemetría real de tráfico de red (`uso-red`)**

El pipeline de S8 (Spark Structured Streaming sobre `red-telemetria`) se observa ahora **desde afuera**: métricas de infraestructura (`kafka-exporter → Prometheus → Grafana`) y métricas de aplicación calculadas en Spark (`isValid`, `processedAt`, `latencyMs`, `lastProgress`), más alertas con umbrales medidos y una estimación de costos.

```text
NIC real ─► agente_red.py ─► MQTT ─► bridge ─► Kafka red-telemetria (3 particiones)
                                                   │                 │
                                     kafka-exporter:9308          readStream (este notebook)
                                                   │                 │  isValid / processedAt / latencyMs
                                         Prometheus:9090             ├─► consola + lastProgress
                                                   │                 └─► Parquet (evidencia)
                                           Grafana:3000                    └─► latencia por minuto, costos
```

## 3.1 Entorno y observabilidad

Desde `D:\bigdataproyectosello\s8bigdata` (PowerShell):

```powershell
docker compose -f kafka/compose.yml up -d        # kafka + kafka-ui + kafka-exporter
docker compose -f uso-red/compose.yml up -d      # mosquitto + uso-red
docker compose -f obs/compose.yml up -d          # prometheus (:49090) + grafana (:43000)
docker compose -f pyspark/compose.yml up -d      # jupyter (:4488), unido a lambda26-kafka-net

docker exec -it lambda26-uso-red python /app/bridge_mqtt_kafka.py   # puente MQTT -> Kafka
docker exec -it lambda26-uso-red python /app/consumer_red.py        # consumer group uso-red-group
python -u uso-red/agente/agente_red.py                               # sensor real (NIC)
```

`obs/` provisiona por archivo el datasource Prometheus, el tablero **S9 - Observabilidad del pipeline** y tres reglas de alerta, para que el entorno sea reproducible.

## 3.2 Prometheus: targets y consultas PromQL reales

El contenedor de Spark está en la misma red que Prometheus, así que la API HTTP (`/api/v1/...`) se consulta directamente desde el notebook.

In [1]:
import json, urllib.request, urllib.parse, datetime, time, os, shutil
LIMA = datetime.timezone(datetime.timedelta(hours=-5))
print("Hora de ejecución (Lima):", datetime.datetime.now(LIMA).strftime("%Y-%m-%d %H:%M:%S"))

PROM = "http://prometheus:9090/api/v1"
def prom(path):
    with urllib.request.urlopen(PROM + path, timeout=10) as r:
        return json.loads(r.read())["data"]
def promql(q):
    return prom("/query?query=" + urllib.parse.quote(q))["result"]

print("\nTARGETS")
for t in prom("/targets")["activeTargets"]:
    print(f"  {t['labels']['job']:<15} {t['scrapeUrl']:<36} {t['health'].upper():<5} último scrape: {t['lastScrape'][:19]}")

Hora de ejecución (Lima): 2026-10-14 18:13:05

TARGETS
  kafka-exporter  http://kafka-exporter:9308/metrics   UP    último scrape: 2026-10-14T23:12:58
  prometheus      http://localhost:9090/metrics        UP    último scrape: 2026-10-14T23:12:49


In [2]:
consultas = [
    "kafka_brokers",
    'up{job="kafka-exporter"}',
    "kafka_broker_info",
    'kafka_topic_partitions{topic="red-telemetria"}',
    'kafka_consumergroup_lag{consumergroup="uso-red-group"}',
    'sum(rate(kafka_topic_partition_current_offset{topic="red-telemetria"}[1m]))',
]
for q in consultas:
    print(f"\n>>> {q}")
    for r in promql(q):
        etiquetas = {k: v for k, v in r["metric"].items() if k not in ("__name__", "instance", "job")}
        print(f"    {etiquetas if etiquetas else ''} = {r['value'][1]}")


>>> kafka_brokers
     = 1

>>> up{job="kafka-exporter"}
     = 1

>>> kafka_broker_info
    {'address': 'kafka:9092', 'id': '1'} = 1

>>> kafka_topic_partitions{topic="red-telemetria"}
    {'topic': 'red-telemetria'} = 3

>>> kafka_consumergroup_lag{consumergroup="uso-red-group"}
    {'consumergroup': 'uso-red-group', 'partition': '0', 'topic': 'red-telemetria'} = 1
    {'consumergroup': 'uso-red-group', 'partition': '1', 'topic': 'red-telemetria'} = 0
    {'consumergroup': 'uso-red-group', 'partition': '2', 'topic': 'red-telemetria'} = 2

>>> sum(rate(kafka_topic_partition_current_offset{topic="red-telemetria"}[1m]))
     = 2.9655098313097668


| Consulta | Esperado | Lectura |
|---|---|---|
| `kafka_brokers` | 1 | El exporter ve el único broker KRaft. |
| `up{job="kafka-exporter"}` | 1 | El último scrape tuvo éxito. |
| `kafka_consumergroup_lag{consumergroup="uso-red-group"}` | cerca de 0 | El consumer de S7 va al día; solo existe si el consumer se conectó alguna vez. |
| `rate(...current_offset...[1m])` | ≈ 3 msg/s | 3 NIC × 1 lectura/s del agente. |

## 3.3 Grafana

El tablero se provisiona desde `obs/grafana/dashboards/s9-observabilidad.json` con 6 paneles: **Kafka Brokers** (stat), **Kafka Exporter Up** (stat), **Kafka Broker Info** (table), **Consumer Lag** (time series), **Throughput red-telemetria** (time series) y **Offset por partición** (time series). Se verifica por API que exista y que el datasource responda:

In [3]:
import base64
AUTH = {"Authorization": "Basic " + base64.b64encode(b"admin:admin").decode()}
def grafana(path):
    req = urllib.request.Request("http://grafana:3000" + path, headers=AUTH)
    with urllib.request.urlopen(req, timeout=10) as r:
        return json.loads(r.read())

d = grafana("/api/dashboards/uid/s9-observabilidad")["dashboard"]
print("Tablero:", d["title"])
for p in d["panels"]:
    print(f"  [{p['type']:<10}] {p['title']:<38} {p['targets'][0]['expr']}")
print("\nDatasource:", grafana("/api/datasources/uid/prometheus/health"))
print("\nReglas de alerta provisionadas:")
for r in grafana("/api/v1/provisioning/alert-rules"):
    print(f"  {r['title']:<36} for={r['for']}  expr={r['data'][0]['model']['expr']}")

Tablero: S9 - Observabilidad del pipeline
  [stat      ] Kafka Brokers                          kafka_brokers
  [stat      ] Kafka Exporter Up                      up{job="kafka-exporter"}
  [table     ] Kafka Broker Info                      kafka_broker_info
  [timeseries] Consumer Lag (uso-red-group)           sum by (consumergroup, partition) (kafka_consumergroup_lag{topic="red-telemetria"})
  [timeseries] Throughput red-telemetria (msg/s)      sum(rate(kafka_topic_partition_current_offset{topic="red-telemetria"}[1m]))
  [timeseries] Offset por partición (red-telemetria)  kafka_topic_partition_current_offset{topic="red-telemetria"}

Datasource: {'details': {'application': 'Prometheus', 'features': {'rulerApiEnabled': False}}, 'message': 'Successfully queried the Prometheus API.', 'status': 'OK'}

Reglas de alerta provisionadas:
  Exporter caido                       for=1m  expr=up{job="kafka-exporter"}
  Kafka no visible                     for=1m  expr=kafka_brokers
  Lag alto so

## 3.4 `SparkSession`

In [4]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName("s09-observabilidad-red")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.sql.session.timeZone", "America/Lima")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1e440694-11fb-4a04-8027-dc39f022dc50;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central


	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central


:: resolution report :: resolve 371ms :: artifacts dl 15ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 from central in [default]
	org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 from central in [default]
	org.slf4j#slf4j-api;2.0.17 from central in [default]
	org.xerial.snappy#snappy-java;1.1.10.8 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf  

26/10/14 23:13:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 4.2.0


## 3.5 Lectura streaming con campos de observabilidad

- `isValid`: el mismo contrato de `consumer_red.py` (S7), ahora en Spark: campos obligatorios no nulos **y** rango físico (tasas ≥ 0 y tráfico ≤ capacidad del enlace × 1,05).
- `processedAt`: instante del micro-lote que procesa el evento (`current_timestamp()`).
- `latencyMs = processedAt − timestamp`: desde que el agente leyó la NIC hasta Spark.
- Extra: `transporteMs = kafkaTimestamp − timestamp` separa la parte de **transporte** (agente → MQTT → bridge → broker) de la **espera al trigger** de Spark.

In [5]:
from pyspark.sql.functions import col, from_json, current_timestamp, unix_millis, lit
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_red = (
    StructType()
    .add("tipoEvento", StringType()).add("sensorId", StringType())
    .add("bytesSentPerSec", DoubleType()).add("bytesRecvPerSec", DoubleType())
    .add("pktsSentPerSec", DoubleType()).add("pktsRecvPerSec", DoubleType())
    .add("errin", LongType()).add("errout", LongType()).add("dropin", LongType()).add("dropout", LongType())
    .add("velocidadEnlaceMbps", DoubleType()).add("origen", StringType()).add("timestamp", LongType())
)

crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "red-telemetria")
    .option("startingOffsets", "latest")
    .load()
)

eventos = (
    crudo.select("topic", "partition", "offset", col("timestamp").alias("kafkaTimestamp"),
                 col("value").cast("string").alias("json_str"))
    .select("topic", "partition", "offset", "kafkaTimestamp", from_json("json_str", esquema_red).alias("e"))
    .select("topic", "partition", "offset", "kafkaTimestamp", "e.*")
)

capacidad = col("velocidadEnlaceMbps") * 125000.0
campos_ok = (col("tipoEvento") == "red.lectura") & col("sensorId").isNotNull() & col("timestamp").isNotNull() \
    & col("bytesSentPerSec").isNotNull() & col("bytesRecvPerSec").isNotNull() & col("velocidadEnlaceMbps").isNotNull()
rango_ok = (col("bytesSentPerSec") >= 0) & (col("bytesRecvPerSec") >= 0) & (col("pktsSentPerSec") >= 0) & (col("pktsRecvPerSec") >= 0) \
    & (col("bytesSentPerSec") <= capacidad * 1.05) & (col("bytesRecvPerSec") <= capacidad * 1.05)

observables = (
    eventos
    .withColumn("isValid", (campos_ok & rango_ok).cast("boolean"))
    .withColumn("processedAt", unix_millis(current_timestamp()))
    .withColumn("latencyMs", col("processedAt") - col("timestamp"))
    .withColumn("transporteMs", unix_millis(col("kafkaTimestamp")) - col("timestamp"))
)
observables.printSchema()

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- bytesSentPerSec: double (nullable = true)
 |-- bytesRecvPerSec: double (nullable = true)
 |-- pktsSentPerSec: double (nullable = true)
 |-- pktsRecvPerSec: double (nullable = true)
 |-- errin: long (nullable = true)
 |-- errout: long (nullable = true)
 |-- dropin: long (nullable = true)
 |-- dropout: long (nullable = true)
 |-- velocidadEnlaceMbps: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = true)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)
 |-- transporteMs: long (nullable = true)



## 3.6 Consola con los campos de observabilidad

Se deja corriendo; en 3.8 arranca en paralelo la consulta a Parquet.

In [6]:
consulta = (
    observables.select("sensorId", "partition", "offset", "bytesRecvPerSec", "isValid", "timestamp", "processedAt", "latencyMs", "transporteMs")
    .writeStream.outputMode("append").format("console")
    .option("truncate", "false").option("numRows", 6)
    .trigger(processingTime="5 seconds")
    .start()
)
time.sleep(17)

-------------------------------------------
Batch: 0
-------------------------------------------
+--------+---------+------+---------------+-------+---------+-----------+---------+------------+
|sensorId|partition|offset|bytesRecvPerSec|isValid|timestamp|processedAt|latencyMs|transporteMs|
+--------+---------+------+---------------+-------+---------+-----------+---------+------------+
+--------+---------+------+---------------+-------+---------+-----------+---------+------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+----------------------------------+---------+------+------------------+-------+-------------+-------------+---------+------------+
|sensorId                          |partition|offset|bytesRecvPerSec   |isValid|timestamp    |processedAt  |latencyMs|transporteMs|
+----------------------------------+---------+------+------------------+-------+-------------+-------------+---------+------------+
|Ethernet 2                        |0        |15002 |0.0               |true   |1792019593527|1792019595167|1640     |444         |
|Ethernet 2                        |0        |15003 |0.0               |true   |1792019594554|1792019595167|613      |396         |
|Ethernet                          |2        |29935 |401.26367412449196|true   |1792019593527|1792019595167|1640     |443         |
|vEthernet (WSL (Hyper-V firewall))|2        |29936 |68.17586696289912 |true   |1792019593527|1792019595167|164

-------------------------------------------
Batch: 2
-------------------------------------------
+----------+---------+------+-----------------+-------+-------------+-------------+---------+------------+
|sensorId  |partition|offset|bytesRecvPerSec  |isValid|timestamp    |processedAt  |latencyMs|transporteMs|
+----------+---------+------+-----------------+-------+-------------+-------------+---------+------------+
|Ethernet 2|0        |15004 |0.0              |true   |1792019595582|1792019600005|4423     |340         |
|Ethernet 2|0        |15005 |0.0              |true   |1792019596615|1792019600005|3390     |275         |
|Ethernet 2|0        |15006 |0.0              |true   |1792019597642|1792019600005|2363     |221         |
|Ethernet 2|0        |15007 |0.0              |true   |1792019598675|1792019600005|1330     |156         |
|Ethernet 2|0        |15008 |0.0              |true   |1792019599704|1792019600005|301      |93          |
|Ethernet  |2        |29939 |1566.503906624918|

-------------------------------------------
Batch: 3
-------------------------------------------
+----------+---------+------+------------------+-------+-------------+-------------+---------+------------+
|sensorId  |partition|offset|bytesRecvPerSec   |isValid|timestamp    |processedAt  |latencyMs|transporteMs|
+----------+---------+------+------------------+-------+-------------+-------------+---------+------------+
|Ethernet 2|0        |15009 |0.0               |true   |1792019600730|1792019605005|4275     |40          |
|Ethernet 2|0        |15010 |0.0               |true   |1792019601763|1792019605005|3242     |-21         |
|Ethernet 2|0        |15011 |0.0               |true   |1792019602795|1792019605005|2210     |-87         |
|Ethernet 2|0        |15012 |0.0               |true   |1792019603821|1792019605005|1184     |-137        |
|Ethernet 2|0        |15013 |0.0               |true   |1792019604847|1792019605005|158      |-182        |
|Ethernet  |2        |29949 |14237.6164

## 3.7 Throughput y *lag* con `lastProgress`

Log estructurado en JSON (mismo estilo que `bridge`/`consumer` de S7). `processedRowsPerSecond > inputRowsPerSecond` es la situación sana. `avgOffsetsBehindLatest` / `maxOffsetsBehindLatest` son el lag visto desde Spark.

In [7]:
def log_progreso(q, topic="red-telemetria"):
    p = q.lastProgress
    if p is None:
        print("Todavía no hay micro-lote procesado."); return None
    m = p["sources"][0].get("metrics", {})
    log = {
        "service": "spark-streaming", "component": "observabilidad", "topic": topic,
        "timestamp": p["timestamp"], "batchId": p["batchId"], "numInputRows": p["numInputRows"],
        "inputRowsPerSecond": round(p["inputRowsPerSecond"], 2),
        "processedRowsPerSecond": round(p["processedRowsPerSecond"], 2),
        "triggerExecutionMs": p["durationMs"].get("triggerExecution"),
        "avgOffsetsBehindLatest": m.get("avgOffsetsBehindLatest"),
        "maxOffsetsBehindLatest": m.get("maxOffsetsBehindLatest"),
        "endOffset": p["sources"][0]["endOffset"],
        "status": "idle" if p["numInputRows"] == 0 else "processed",
    }
    print(json.dumps(log, indent=2))
    return log

progreso_1 = log_progreso(consulta)

{
  "service": "spark-streaming",
  "component": "observabilidad",
  "topic": "red-telemetria",
  "timestamp": "2026-10-14T23:13:25.000Z",
  "batchId": 3,
  "numInputRows": 15,
  "inputRowsPerSecond": 3.0,
  "processedRowsPerSecond": 23.1,
  "triggerExecutionMs": 650,
  "avgOffsetsBehindLatest": "0.0",
  "maxOffsetsBehindLatest": "0",
  "endOffset": "{\"red-telemetria\": {\"0\": 15014, \"1\": 0, \"2\": 29959}}",
  "status": "processed"
}


## 3.8 Persistir la evidencia observable en Parquet

Corre en paralelo con la consola durante **6 minutos** para tener varias ventanas de un minuto. A mitad de la corrida se publica un evento **inválido** (tasa negativa, como un contador reiniciado) para comprobar que `isValid` lo marca. El conteo y el tamaño en disco se miden en dos momentos para estimar el crecimiento por hora.

In [8]:
from pyspark.sql.functions import to_json, struct
ARTIFACTS = "/opt/artifacts/s09_red_observabilidad"
CHECKPOINT_PARQUET = "/opt/artifacts/s09_chk_red_observabilidad"
shutil.rmtree(ARTIFACTS, ignore_errors=True); shutil.rmtree(CHECKPOINT_PARQUET, ignore_errors=True)

def tamano_mb(ruta):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(ruta) for f in fs if f.endswith(".parquet")) / 1e6

def filas():
    try: return spark.read.parquet(ARTIFACTS).count()
    except Exception: return 0

consulta2 = (
    observables.writeStream.outputMode("append").format("parquet")
    .option("path", ARTIFACTS).option("checkpointLocation", CHECKPOINT_PARQUET)
    .trigger(processingTime="5 seconds")
    .start()
)

muestras_progreso = []
t_inicio = time.time()
mediciones = []
invalido_publicado = False
while time.time() - t_inicio < 360:
    time.sleep(30)
    if consulta2.lastProgress: muestras_progreso.append(consulta2.lastProgress)
    mediciones.append((round(time.time() - t_inicio), filas(), round(tamano_mb(ARTIFACTS), 4)))
    if not invalido_publicado and time.time() - t_inicio > 150:
        cols = ["tipoEvento", "sensorId", "bytesSentPerSec", "bytesRecvPerSec", "pktsSentPerSec", "pktsRecvPerSec",
                "errin", "errout", "dropin", "dropout", "velocidadEnlaceMbps", "origen", "timestamp"]
        df = spark.createDataFrame([("red.lectura", "Ethernet", -1500.0, 200.0, 3.0, 3.0, 0, 0, 0, 0, 1000.0, "prueba-s09", int(time.time()*1000))], cols)
        df.select(col("sensorId").alias("key"), to_json(struct(*cols)).alias("value")).write.format("kafka") \
          .option("kafka.bootstrap.servers", "kafka:9092").option("topic", "red-telemetria").save()
        invalido_publicado = True
        print(">>> publicado evento inválido (bytesSentPerSec = -1500.0, origen = prueba-s09)")

progreso_2 = log_progreso(consulta)
consulta.stop()
consulta2.stop()

print("\nsegundos | filas en Parquet | MB en disco")
for s, n, mb in mediciones:
    print(f"{s:>8} | {n:>16} | {mb:>10}")

-------------------------------------------
Batch: 4
-------------------------------------------
+----------+---------+------+------------------+-------+-------------+-------------+---------+------------+
|sensorId  |partition|offset|bytesRecvPerSec   |isValid|timestamp    |processedAt  |latencyMs|transporteMs|
+----------+---------+------+------------------+-------+-------------+-------------+---------+------------+
|Ethernet 2|0        |15014 |0.0               |true   |1792019605877|1792019610004|4127     |-232        |
|Ethernet 2|0        |15015 |0.0               |true   |1792019606906|1792019610004|3098     |-283        |
|Ethernet 2|0        |15016 |0.0               |true   |1792019607933|1792019610004|2071     |-333        |
|Ethernet 2|0        |15017 |0.0               |true   |1792019608959|1792019610004|1045     |-383        |
|Ethernet 2|0        |15018 |0.0               |true   |1792019609983|1792019610004|21       |-429        |
|Ethernet  |2        |29959 |203.925511

-------------------------------------------
Batch: 5
-------------------------------------------
+----------------------------------+---------+------+------------------+-------+-------------+-------------+---------+------------+
|sensorId                          |partition|offset|bytesRecvPerSec   |isValid|timestamp    |processedAt  |latencyMs|transporteMs|
+----------------------------------+---------+------+------------------+-------+-------------+-------------+---------+------------+
|Ethernet 2                        |0        |15019 |0.0               |true   |1792019611009|1792019615004|3995     |-477        |
|Ethernet 2                        |0        |15020 |0.0               |true   |1792019612036|1792019615004|2968     |-526        |
|Ethernet 2                        |0        |15021 |0.0               |true   |1792019613062|1792019615004|1942     |-572        |
|Ethernet                          |2        |29969 |1119.6124371456528|true   |1792019611009|1792019615004|399

-------------------------------------------
Batch: 6
-------------------------------------------
+----------+---------+------+---------------+-------+-------------+-------------+---------+------------+
|sensorId  |partition|offset|bytesRecvPerSec|isValid|timestamp    |processedAt  |latencyMs|transporteMs|
+----------+---------+------+---------------+-------+-------------+-------------+---------+------------+
|Ethernet 2|0        |15022 |0.0            |true   |1792019614089|1792019620005|5916     |965         |
|Ethernet 2|0        |15023 |0.0            |true   |1792019615116|1792019620005|4889     |919         |
|Ethernet 2|0        |15024 |0.0            |true   |1792019616145|1792019620005|3860     |868         |
|Ethernet 2|0        |15025 |0.0            |true   |1792019617170|1792019620005|2835     |824         |
|Ethernet 2|0        |15026 |0.0            |true   |1792019618198|1792019620005|1807     |775         |
|Ethernet 2|0        |15027 |0.0            |true   |1792019619

-------------------------------------------
Batch: 7
-------------------------------------------
+----------+---------+------+-----------------+-------+-------------+-------------+---------+------------+
|sensorId  |partition|offset|bytesRecvPerSec  |isValid|timestamp    |processedAt  |latencyMs|transporteMs|
+----------+---------+------+-----------------+-------+-------------+-------------+---------+------------+
|Ethernet 2|0        |15028 |0.0              |true   |1792019620251|1792019625004|4753     |679         |
|Ethernet 2|0        |15029 |0.0              |true   |1792019621276|1792019625004|3728     |632         |
|Ethernet 2|0        |15030 |0.0              |true   |1792019622301|1792019625004|2703     |591         |
|Ethernet 2|0        |15031 |0.0              |true   |1792019623333|1792019625004|1671     |537         |
|Ethernet 2|0        |15032 |0.0              |true   |1792019624360|1792019625004|644      |490         |
|Ethernet  |2        |29987 |187.0454389308323|

-------------------------------------------
Batch: 8
-------------------------------------------
+----------+---------+------+----------------+-------+-------------+-------------+---------+------------+
|sensorId  |partition|offset|bytesRecvPerSec |isValid|timestamp    |processedAt  |latencyMs|transporteMs|
+----------+---------+------+----------------+-------+-------------+-------------+---------+------------+
|Ethernet 2|0        |15033 |0.0             |true   |1792019625387|1792019630005|4618     |441         |
|Ethernet 2|0        |15034 |0.0             |true   |1792019626413|1792019630005|3592     |393         |
|Ethernet 2|0        |15035 |0.0             |true   |1792019627439|1792019630005|2566     |347         |
|Ethernet 2|0        |15036 |0.0             |true   |1792019628465|1792019630005|1540     |299         |
|Ethernet 2|0        |15037 |0.0             |true   |1792019629492|1792019630005|513      |260         |
|Ethernet  |2        |29997 |2639.23413531174|true   |1

...

-------------------------------------------
Batch: 76
-------------------------------------------
+----------+---------+------+-----------------+-------+-------------+-------------+---------+------------+
|sensorId  |partition|offset|bytesRecvPerSec  |isValid|timestamp    |processedAt  |latencyMs|transporteMs|
+----------+---------+------+-----------------+-------+-------------+-------------+---------+------------+
|Ethernet 2|0        |15363 |0.0              |true   |1792019964924|1792019970003|5079     |781         |
|Ethernet 2|0        |15364 |0.0              |true   |1792019965950|1792019970003|4053     |733         |
|Ethernet 2|0        |15365 |0.0              |true   |1792019966977|1792019970003|3026     |680         |
|Ethernet 2|0        |15366 |0.0              |true   |1792019968002|1792019970003|2001     |631         |
|Ethernet 2|0        |15367 |0.0              |true   |1792019969027|1792019970003|976      |582         |
|Ethernet  |2        |30658 |4262.395302822028

-------------------------------------------
Batch: 77
-------------------------------------------
+----------+---------+------+------------------+-------+-------------+-------------+---------+------------+
|sensorId  |partition|offset|bytesRecvPerSec   |isValid|timestamp    |processedAt  |latencyMs|transporteMs|
+----------+---------+------+------------------+-------+-------------+-------------+---------+------------+
|Ethernet 2|0        |15368 |0.0               |true   |1792019970053|1792019975003|4950     |534         |
|Ethernet 2|0        |15369 |0.0               |true   |1792019971081|1792019975003|3922     |483         |
|Ethernet 2|0        |15370 |0.0               |true   |1792019972107|1792019975003|2896     |436         |
|Ethernet 2|0        |15371 |0.0               |true   |1792019973137|1792019975003|1866     |383         |
|Ethernet 2|0        |15372 |0.0               |true   |1792019974163|1792019975003|840      |331         |
|Ethernet  |2        |30668 |155.00861

-------------------------------------------
Batch: 78
-------------------------------------------
+----------+---------+------+------------------+-------+-------------+-------------+---------+------------+
|sensorId  |partition|offset|bytesRecvPerSec   |isValid|timestamp    |processedAt  |latencyMs|transporteMs|
+----------+---------+------+------------------+-------+-------------+-------------+---------+------------+
|Ethernet 2|0        |15373 |0.0               |true   |1792019975188|1792019980004|4816     |279         |
|Ethernet 2|0        |15374 |0.0               |true   |1792019976214|1792019980004|3790     |228         |
|Ethernet 2|0        |15375 |0.0               |true   |1792019977241|1792019980004|2763     |175         |
|Ethernet 2|0        |15376 |0.0               |true   |1792019978268|1792019980004|1736     |123         |
|Ethernet 2|0        |15377 |0.0               |true   |1792019979294|1792019980004|710      |71          |
|Ethernet  |2        |30678 |3100.4904

-------------------------------------------
Batch: 79
-------------------------------------------
+----------+---------+------+-----------------+-------+-------------+-------------+---------+------------+
|sensorId  |partition|offset|bytesRecvPerSec  |isValid|timestamp    |processedAt  |latencyMs|transporteMs|
+----------+---------+------+-----------------+-------+-------------+-------------+---------+------------+
|Ethernet 2|0        |15378 |0.0              |true   |1792019980321|1792019985005|4684     |18          |
|Ethernet 2|0        |15379 |0.0              |true   |1792019981347|1792019985005|3658     |-34         |
|Ethernet 2|0        |15380 |0.0              |true   |1792019982373|1792019985005|2632     |-85         |
|Ethernet 2|0        |15381 |0.0              |true   |1792019983401|1792019985005|1604     |-139        |
|Ethernet 2|0        |15382 |0.0              |true   |1792019984431|1792019985005|574      |-198        |
|Ethernet  |2        |30688 |198.6626428201598

{
  "service": "spark-streaming",
  "component": "observabilidad",
  "topic": "red-telemetria",
  "timestamp": "2026-10-14T23:19:45.001Z",
  "batchId": 79,
  "numInputRows": 15,
  "inputRowsPerSecond": 3.0,
  "processedRowsPerSecond": 64.7,
  "triggerExecutionMs": 232,
  "avgOffsetsBehindLatest": "0.0",
  "maxOffsetsBehindLatest": "0",
  "endOffset": "{\"red-telemetria\": {\"0\": 15383, \"1\": 0, \"2\": 30698}}",
  "status": "processed"
}

segundos | filas en Parquet | MB en disco
      32 |               84 |     0.0835
      64 |              171 |     0.1785
      97 |              273 |     0.2601
     131 |              360 |      0.355
     164 |              462 |     0.4501
     199 |              562 |     0.5453
     233 |              664 |     0.6267
     267 |              766 |     0.7218
     302 |              871 |     0.8171
     337 |              973 |     0.9121
     372 |             1075 |     1.0208


In [9]:
guardado = spark.read.parquet(ARTIFACTS)
print("filas guardadas:", guardado.count())
guardado.groupBy("isValid").count().show()
guardado.filter(~col("isValid")).select("sensorId", "bytesSentPerSec", "origen", "isValid", "latencyMs").show(truncate=False)
guardado.select("sensorId", "partition", "offset", "isValid", "processedAt", "latencyMs", "transporteMs").orderBy(col("offset").desc()).show(8, truncate=False)

filas guardadas: 1090


+-------+-----+
|isValid|count|
+-------+-----+
|  false|    1|
|   true| 1089|
+-------+-----+



+--------+---------------+----------+-------+---------+
|sensorId|bytesSentPerSec|origen    |isValid|latencyMs|
+--------+---------------+----------+-------+---------+
|Ethernet|-1500.0        |prueba-s09|false  |3378     |
+--------+---------------+----------+-------+---------+



+----------------------------------+---------+------+-------+-------------+---------+------------+
|sensorId                          |partition|offset|isValid|processedAt  |latencyMs|transporteMs|
+----------------------------------+---------+------+-------+-------------+---------+------------+
|vEthernet (WSL (Hyper-V firewall))|2        |30697 |true   |1792019985002|571      |-197        |
|Ethernet                          |2        |30696 |true   |1792019985002|571      |-199        |
|vEthernet (WSL (Hyper-V firewall))|2        |30695 |true   |1792019985002|1601     |-138        |
|Ethernet                          |2        |30694 |true   |1792019985002|1601     |-140        |
|vEthernet (WSL (Hyper-V firewall))|2        |30693 |true   |1792019985002|2629     |-84         |
|Ethernet                          |2        |30692 |true   |1792019985002|2629     |-86         |
|vEthernet (WSL (Hyper-V firewall))|2        |30691 |true   |1792019985002|3655     |-33         |
|Ethernet 

## 3.9 Latencia agregada por ventana de un minuto (batch sobre el Parquet)

In [10]:
from pyspark.sql.functions import window, avg, min as spark_min, max as spark_max, count, percentile_approx, round as rnd, sum as spark_sum

latencia_por_minuto = (
    guardado.groupBy(window(col("kafkaTimestamp"), "1 minute"))
    .agg(count("*").alias("numEventos"),
         spark_sum((~col("isValid")).cast("int")).alias("invalidos"),
         rnd(avg("latencyMs"), 1).alias("avgLatencyMs"),
         spark_min("latencyMs").alias("minLatencyMs"),
         percentile_approx("latencyMs", 0.95).alias("p95LatencyMs"),
         spark_max("latencyMs").alias("maxLatencyMs"),
         rnd(avg("transporteMs"), 1).alias("avgTransporteMs"))
    .select(col("window.start").alias("inicioVentana"), col("window.end").alias("finVentana"),
            "numEventos", "invalidos", "avgLatencyMs", "minLatencyMs", "p95LatencyMs", "maxLatencyMs", "avgTransporteMs")
    .orderBy("inicioVentana")
)
latencia_por_minuto.show(20, truncate=False)
filas_lat = latencia_por_minuto.collect()
completas = [r for r in filas_lat if r["numEventos"] >= 150]   # minutos completos (≈180 eventos)
global_lat = guardado.agg(avg("latencyMs").alias("avg"), percentile_approx("latencyMs", 0.95).alias("p95"),
                          spark_max("latencyMs").alias("max"), spark_min("latencyMs").alias("min"),
                          avg("transporteMs").alias("transp")).first()
print("Global -> avg %.0f ms | p95 %d ms | min %d ms | max %d ms | transporte medio %.0f ms"
      % (global_lat["avg"], global_lat["p95"], global_lat["min"], global_lat["max"], global_lat["transp"]))

+-------------------+-------------------+----------+---------+------------+------------+------------+------------+---------------+
|inicioVentana      |finVentana         |numEventos|invalidos|avgLatencyMs|minLatencyMs|p95LatencyMs|maxLatencyMs|avgTransporteMs|
+-------------------+-------------------+----------+---------+------------+------------+------------+------------+---------------+
|2026-10-14 18:13:00|2026-10-14 18:14:00|84        |0        |2671.8      |222         |4889        |5916        |303.1          |
|2026-10-14 18:14:00|2026-10-14 18:15:00|174       |0        |2663.8      |-262        |5452        |5669        |241.5          |
|2026-10-14 18:15:00|2026-10-14 18:16:00|174       |0        |2741.0      |-92         |5053        |5799        |229.5          |
|2026-10-14 18:16:00|2026-10-14 18:17:00|175       |1        |2773.5      |201         |5069        |5300        |224.9          |
|2026-10-14 18:17:00|2026-10-14 18:18:00|175       |0        |2817.4      |-364    

Global -> avg 2702 ms | p95 5053 ms | min -364 ms | max 5916 ms | transporte medio 220 ms


**Por qué min y max varían tanto con el sistema sano:** el agente publica cada **1000 ms** y Spark dispara cada **5000 ms**. Un evento espera hasta el próximo disparo entre ~0 s (llegó justo antes) y ~5 s (llegó justo después), así que la latencia por evento se reparte casi uniforme entre el transporte (~decenas de ms) y ~5 s, con un promedio cercano a 2,5 s. Como 5 es múltiplo de 1, el patrón se repite en cada micro-lote (≈15 eventos: 5 rondas × 3 NIC). El `transporteMs` muestra que casi toda la latencia es **espera al trigger**, no red.

## 3.10 Alertas propuestas y umbrales (calculados con los datos medidos)

In [11]:
eps = [p["inputRowsPerSecond"] for p in muestras_progreso if p["numInputRows"] > 0]
eps_prom = sum(eps) / len(eps)
eps_pico = max(eps)
max_avg_min = max(r["avgLatencyMs"] for r in completas)
lag_actual = sum(float(r["value"][1]) for r in promql('kafka_consumergroup_lag{consumergroup="uso-red-group"}'))
lag_max_30m = promql('max_over_time(sum(kafka_consumergroup_lag{consumergroup="uso-red-group"})[30m:15s])')
lag_max_30m = float(lag_max_30m[0]["value"][1]) if lag_max_30m else None

umbral_lag = round(eps_pico * 20)                                  # 20 s de lecturas pendientes
umbral_lat_sensible = int(round(max_avg_min * 1.5, -2))            # 50 % sobre el peor minuto medido
umbral_lat_critica = 10000                                         # 2 triggers completos sin procesar

alertas = [
    ("Kafka no visible", "kafka_brokers < 1 (for 1m)", "Prometheus/Grafana", "Valor medido = 1 siempre; cualquier 0 es una caída real."),
    ("Exporter caído", 'up{job="kafka-exporter"} == 0 (for 1m)', "Prometheus/Grafana", "Binario; probado deteniendo el exporter."),
    ("Lag alto sostenido", f"sum(kafka_consumergroup_lag{{consumergroup=\"uso-red-group\"}}) > {umbral_lag} (for 2m)", "Prometheus/Grafana",
     f"Pico de entrada {eps_pico:.1f} ev/s × 20 s. Lag medido: actual {lag_actual:.0f}, máximo 30 min {lag_max_30m:.0f}."),
    ("Latencia alta sensible", f"avgLatencyMs > {umbral_lat_sensible} por minuto", "Spark / Parquet",
     f"Peor minuto medido: {max_avg_min:.0f} ms (+50 %)."),
    ("Latencia alta crítica", f"avgLatencyMs > {umbral_lat_critica} por minuto", "Spark / Parquet",
     "2 triggers de 5 s: Spark dejó de seguir el ritmo."),
    ("Lecturas inválidas", "invalidos / numEventos > 1 % por minuto", "Spark / Parquet",
     "En operación normal 0 inválidos; 1 % ≈ 2 lecturas/min ya indica contadores reiniciados o un agente defectuoso."),
]
print(f"{'Situación':<24} | {'Regla':<70} | {'Dónde':<18} | Justificación")
print("-" * 180)
for a in alertas:
    print(f"{a[0]:<24} | {a[1]:<70} | {a[2]:<18} | {a[3]}")

Situación                | Regla                                                                  | Dónde              | Justificación
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Kafka no visible         | kafka_brokers < 1 (for 1m)                                             | Prometheus/Grafana | Valor medido = 1 siempre; cualquier 0 es una caída real.
Exporter caído           | up{job="kafka-exporter"} == 0 (for 1m)                                 | Prometheus/Grafana | Binario; probado deteniendo el exporter.
Lag alto sostenido       | sum(kafka_consumergroup_lag{consumergroup="uso-red-group"}) > 60 (for 2m) | Prometheus/Grafana | Pico de entrada 3.0 ev/s × 20 s. Lag medido: actual 12, máximo 30 min 15.
Latencia alta sensible   | avgLatencyMs > 4200 por minuto                                         | Spark / Parquet    | Peor minuto medido: 2817 

## 3.11 Estimación de costos y plan de escalado

Precios de referencia de la guía (AWS us-east-1, octubre 2026): `t3.medium` ≈ USD 0,0416/h; S3 Standard ≈ USD 0,023 por GB-mes.

In [12]:
(s1, n1, mb1), (s2, n2, mb2) = mediciones[0], mediciones[-1]
filas_hora = (n2 - n1) / (s2 - s1) * 3600
mb_hora = (mb2 - mb1) / (s2 - s1) * 3600
gb_mes = mb_hora * 730 / 1000
PRECIO_T3 = 0.0416
PRECIO_S3 = 0.023
instancias = 3                                   # kafka+exporter | spark streaming | prometheus+grafana
computo_mes = instancias * PRECIO_T3 * 730
s3_mes = gb_mes * PRECIO_S3
# retención: el Parquet acumulado sin limpieza crece cada mes
s3_12m_sin_retencion = sum(gb_mes * m * PRECIO_S3 for m in range(1, 13)) / 12

plantilla = [
    ("Eventos por segundo (promedio / pico)", f"{eps_prom:.2f} / {eps_pico:.2f} ev/s", "de lastProgress.inputRowsPerSecond"),
    ("Latencia promedio por minuto (peor minuto)", f"{global_lat['avg']:.0f} ms ({max_avg_min:.0f} ms)", "3.9; p95 global " + str(global_lat['p95']) + " ms"),
    ("Crecimiento del Parquet por hora", f"{filas_hora:,.0f} filas/h | {mb_hora:.2f} MB/h", f"medido entre t={s1}s y t={s2}s"),
    ("Almacenamiento por mes", f"{gb_mes:.2f} GB/mes", f"S3 Standard ≈ US$ {s3_mes:.3f}/mes"),
    ("Cómputo por hora", f"{instancias} × t3.medium = US$ {instancias*PRECIO_T3:.4f}/h", f"US$ {computo_mes:.2f}/mes (730 h)"),
    ("Total mensual base", f"US$ {computo_mes + s3_mes:.2f}/mes", "sin tráfico de red ni operación"),
    ("S3 promedio año 1 sin retención", f"US$ {s3_12m_sin_retencion:.3f}/mes", "el Parquet acumulado crece lineal"),
]
print(f"{'Variable':<44} | {'Valor medido / calculado':<40} | Nota")
print("-" * 130)
for v in plantilla:
    print(f"{v[0]:<44} | {v[1]:<40} | {v[2]}")

capacidad_spark = sum(p["processedRowsPerSecond"] for p in muestras_progreso if p["numInputRows"] > 0) / max(len(eps), 1)
print(f"\nCapacidad observada de Spark: {capacidad_spark:.1f} filas/s procesadas vs {eps_prom:.2f} ev/s de entrada "
      f"-> margen x{capacidad_spark/eps_prom:.0f}")
resumen = dict(eps_prom=eps_prom, eps_pico=eps_pico, lat_avg=global_lat["avg"], lat_p95=global_lat["p95"], max_avg_min=max_avg_min,
               filas_hora=filas_hora, mb_hora=mb_hora, gb_mes=gb_mes, computo_mes=computo_mes, s3_mes=s3_mes,
               umbral_lag=umbral_lag, umbral_lat_sensible=umbral_lat_sensible, capacidad_spark=capacidad_spark,
               lag_actual=lag_actual, lag_max_30m=lag_max_30m)
print(json.dumps({k: round(v, 3) if isinstance(v, float) else v for k, v in resumen.items()}))

Variable                                     | Valor medido / calculado                 | Nota
----------------------------------------------------------------------------------------------------------------------------------
Eventos por segundo (promedio / pico)        | 2.93 / 3.00 ev/s                         | de lastProgress.inputRowsPerSecond
Latencia promedio por minuto (peor minuto)   | 2702 ms (2817 ms)                        | 3.9; p95 global 5053 ms
Crecimiento del Parquet por hora             | 10,493 filas/h | 9.92 MB/h               | medido entre t=32s y t=372s
Almacenamiento por mes                       | 7.24 GB/mes                              | S3 Standard ≈ US$ 0.167/mes
Cómputo por hora                             | 3 × t3.medium = US$ 0.1248/h             | US$ 91.10/mes (730 h)
Total mensual base                           | US$ 91.27/mes                            | sin tráfico de red ni operación
S3 promedio año 1 sin retención              | US$ 1.083/mes     

**Plan de escalado (umbrales de decisión):**

1. **Lag de `uso-red-group` > umbral durante 10 min** o `maxOffsetsBehindLatest` de Spark creciendo en 3 lotes seguidos → el consumidor no da abasto: primero más paralelismo del consumidor (más instancias en el grupo, hasta 3 = particiones) y, si no alcanza, más particiones del topic.
2. **`avgLatencyMs` por minuto > umbral sensible sostenido** con `processedRowsPerSecond` cayendo hacia `inputRowsPerSecond` → Spark se satura: subir núcleos (`local[*]` → instancia más grande) o bajar el trigger solo si la duración de lote lo permite.
3. **Volumen ×2 (≈6 ev/s):** con la capacidad observada de Spark (decenas de filas/s por lote) y el broker casi ocioso, lo primero que crece es el **almacenamiento**: Parquet ×2 por mes; se escala aplicando retención/compactación antes que cómputo.